# Light Training: Few-Shot Learning

**Thiết lập tập theo tập (episodic).** Với bài toán \\(N\\)-way \\(K\\)-shot, trong mỗi tập:

- **Tập hỗ trợ (support)**: \\( \mathcal{S}=\{(x_{c,i}^{\mathrm{s}},\, y_{c}^{\mathrm{s}}=c)\}_{c=1..N,\, i=1..K} \\)
- **Tập truy vấn (query)**: \\( \mathcal{Q}=\{(x_{c,j}^{\mathrm{q}},\, y_{c}^{\mathrm{q}}=c)\}_{c=1..N,\, j=1..Q} \\)

Gọi **bộ mã hoá** (encoder) gọn nhẹ là \\( f_\theta:\mathbb{R}^{3\times32\times32}\to\mathbb{R}^d \\),
\\( z=f_\theta(x)\in\mathbb{R}^d \\).
Chuẩn hoá cosine: \\( \hat{z}=z/\|z\|_2 \\).

**Độ đo/điểm tương tự** giữa hai đặc trưng \\(u,v\\):

- **Khoảng cách Euclid**: \\( d_{\mathrm{E}}(u,v)=\|u-v\|_2 \\)
- **Cosine**: \\( s_{\cos}(u,v)=\frac{u^\top v}{\|u\|_2\|v\|_2}=\hat{u}^\top \hat{v} \\)

**Xác suất softmax có nhiệt độ** \\( \tau>0 \\) với các “logits” \\( \{\ell_c\} \\):
\\[
\mathrm{softmax}_\tau(\ell)_c=\frac{\exp(\ell_c/\tau)}{\sum_{c'}\exp(\ell_{c'}/\tau)}.
\\]

**Độ chính xác Top-k** cho tập \\( \mathcal{A} \\):
\\[
\mathrm{Top}@k(\mathcal{A})=\frac{1}{|\mathcal{A}|}\sum_{(x,y)\in\mathcal{A}} \mathbf{1}\{y \in \text{k lớp có xác suất cao nhất cho } x\}.
\\]


## (A) Prototypical Networks

**Prototype mỗi lớp** \\( c \\):
\\[
\mu_c \;=\; \frac{1}{K}\sum_{i=1}^K f_\theta\!\left(x_{c,i}^{\mathrm{s}}\right) \in \mathbb{R}^d.
\\]

**Logits & phân phối dự đoán cho truy vấn** \\( x^{\mathrm{q}} \\) (hai biến thể):

- **Proto-Euclidean** (dùng khoảng cách âm):
\\[
\ell_c(x^{\mathrm{q}}) \;=\; -\, d_{\mathrm{E}}\!\left(f_\theta(x^{\mathrm{q}}),\, \mu_c\right),
\qquad
p_\theta\!\left(y=c\mid x^{\mathrm{q}}\right)\;=\;\mathrm{softmax}_\tau\!\left(\{\ell_c\}\right).
\\]

- **Proto-Cosine** (cosine + hệ số tỉ lệ \\( \alpha \\)):
\\[
\ell_c(x^{\mathrm{q}}) \;=\; \alpha \cdot s_{\cos}\!\left(f_\theta(x^{\mathrm{q}}),\, \mu_c\right),
\qquad
p_\theta\!\left(y=c\mid x^{\mathrm{q}}\right)\;=\;\mathrm{softmax}_\tau\!\left(\{\ell_c\}\right).
\\]

**Hàm mất mát theo tập (episodic cross-entropy)**:
\\[
\mathcal{L}_{\text{proto}}(\theta)
\;=\; - \frac{1}{NQ} \sum_{c=1}^N \sum_{j=1}^Q 
\log p_\theta\!\left(y=c \mid x_{c,j}^{\mathrm{q}}\right).
\\]

**Đánh giá:** Top-1/Top-5 trên \\( \mathcal{Q} \\) (val/test).  
**Checkpoint:** `fsl_prototypical_networks_eucl.pth`, `fsl_prototypical_networks_cos.pth`.


In [1]:
# ========= FEWSHOT PROTO: SHARED =========
import os, csv, math, time, random, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, Sampler
from torchvision import datasets, transforms
from pathlib import Path

# --- device & folders ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKPT_DIR = Path("checkpoints"); CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR  = Path("logs"); LOG_DIR.mkdir(parents=True, exist_ok=True)

# --- CIFAR-10 (supervised split, dùng cho episodic sampling) ---
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2470, 0.2435, 0.2616)
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_full = datasets.CIFAR10("./data", train=True,  download=True, transform=tf)
test_set   = datasets.CIFAR10("./data", train=False, download=True, transform=tf)

# tách 5k ảnh từ train_full làm validation
torch.manual_seed(0)
perm = torch.randperm(len(train_full))
val_idx = perm[:5000]; tr_idx = perm[5000:]
train_set = Subset(train_full, tr_idx.tolist())
val_set   = Subset(train_full, val_idx.tolist())

NUM_CLASSES = 10

# --- encoder gọn nhẹ f_θ ---
class Conv4(nn.Module):
    def __init__(self, out_dim=64):
        super().__init__()
        def block(i,o): 
            return nn.Sequential(
                nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(True), nn.MaxPool2d(2)
            )
        self.enc = nn.Sequential(block(3,32), block(32,64), block(64,64), block(64,out_dim))
        self.out_dim = out_dim
    def forward(self, x):                 # [B,3,32,32] → [B,d]
        return self.enc(x).mean(dim=[2,3])

# --- Episodic sampler N-way K-shot + Q query/way ---
class EpisodeSampler(Sampler):
    def __init__(self, base_set, n_way=5, k_shot=5, q_query=15, episodes_per_epoch=100):
        if isinstance(base_set, Subset):
            targets_full = np.array(base_set.dataset.targets)
            targets = targets_full[base_set.indices]
        else:
            targets = np.array(getattr(base_set, 'targets'))
        self.base = base_set
        self.targets = targets
        self.n_way, self.k_shot, self.q_query = n_way, k_shot, q_query
        self.episodes_per_epoch = episodes_per_epoch
        self.class2idx = {c: np.where(self.targets==c)[0] for c in np.unique(self.targets)}
    def __len__(self):
        return self.episodes_per_epoch * self.n_way * (self.k_shot + self.q_query)
    def __iter__(self):
        for _ in range(self.episodes_per_epoch):
            classes = np.random.choice(list(self.class2idx.keys()), self.n_way, replace=False)
            ep_idx=[]
            for c in classes:
                pool = self.class2idx[c]
                if len(pool) < self.k_shot + self.q_query:
                    pool = np.random.choice(pool, self.k_shot + self.q_query, replace=True)
                else:
                    pool = np.random.permutation(pool)[:self.k_shot + self.q_query]
                ep_idx += pool.tolist()
            yield from ep_idx

def make_episode_loader(base_set, n_way, k_shot, q_query, episodes, num_workers=2):
    sampler = EpisodeSampler(base_set, n_way=n_way, k_shot=k_shot, q_query=q_query, episodes_per_epoch=episodes)
    bs = n_way*(k_shot+q_query)
    return DataLoader(base_set, batch_size=bs, sampler=sampler, num_workers=num_workers, pin_memory=True)

# --- tách support/query theo thứ tự sampler ---
def split_support_query(emb, n_way, k_shot, q_query, device):
    protos=[]; queries=[]; yq=[]
    for c in range(n_way):
        s = c*(k_shot+q_query)
        support = emb[s:s+k_shot]                      # [K,d]
        query   = emb[s+k_shot:s+k_shot+q_query]       # [Q,d]
        protos.append(support.mean(0, keepdim=True))
        queries.append(query)
        yq.append(torch.full((q_query,), c, device=device, dtype=torch.long))
    return torch.cat(protos,0), torch.cat(queries,0), torch.cat(yq,0)  # [N,d], [NQ,d], [NQ]

# --- công thức logits ---
def logits_proto_euclidean(Q, P, tau=1.0):             # ℓ_c = -||q-μ_c||_2 ; softmax_τ
    d = torch.cdist(Q, P)                               # [NQ, N]
    return -d / tau
def logits_proto_cosine(Q, P, alpha=10.0, tau=1.0):
    q = F.normalize(Q, dim=1); p = F.normalize(P, dim=1)
    return (alpha * (q @ p.t())) / tau

# --- eval episodic Top-1/Top-5 ---
@torch.no_grad()
def eval_episodic_topk_proto(encoder, base_set, n_way, k_shot, q_query, episodes, variant='eucl', alpha=10.0, tau=1.0):
    encoder = encoder.to(device).eval()
    loader = make_episode_loader(base_set, n_way, k_shot, q_query, episodes)
    accs1=[]; accs5=[]
    topk = min(5, n_way)
    for xs, _ in loader:
        xs = xs.to(device)
        emb = encoder(xs)
        P, Q, yq = split_support_query(emb, n_way, k_shot, q_query, device)
        if variant=='eucl': logits = logits_proto_euclidean(Q, P, tau=tau)
        else:               logits = logits_proto_cosine(Q, P, alpha=alpha, tau=tau)
        _, pred = logits.topk(topk, 1, True, True)
        correct = pred.eq(yq.view(-1,1))
        accs1.append(100.0*correct[:, :1].sum().item()/yq.size(0))
        accs5.append(100.0*correct[:, :topk].sum().item()/yq.size(0))
    return float(np.mean(accs1)), float(np.mean(accs5))

def write_csv_header(path, header):
    with open(path, "w", newline="") as f: csv.writer(f).writerow(header)
def write_csv_row(path, row):
    with open(path, "a", newline="") as f: csv.writer(f).writerow(row)


100%|██████████| 170M/170M [00:02<00:00, 82.6MB/s] 


In [2]:
# ========= FEWSHOT PROTO: TRAINING (Euclidean & Cosine) =========
def train_prototypical(
    variant='eucl',                 # 'eucl' | 'cos'
    n_way=5, k_shot=5, q_query=15,
    episodes_per_epoch=200,
    epochs=20, enc_dim=64, alpha=10.0, tau=1.0,
    best_by='val@1',               # 'val@1' | 'val@5'
    log_test_each_epoch=True
):
    assert variant in ['eucl','cos']
    label = "fsl_prototypical_networks_eucl" if variant=='eucl' else "fsl_prototypical_networks_cos"
    ckpt  = CKPT_DIR / f"{label}.pth"
    log_csv = LOG_DIR / f"{label}.csv"
    write_csv_header(log_csv, ["epoch","train_loss","val_top1","val_top5","test_top1","test_top5","best_by","is_best"])

    encoder = Conv4(out_dim=enc_dim).to(device)
    opt = torch.optim.Adam(encoder.parameters(), lr=1e-3)

    best_score = None

    print(f"[Proto-{variant}] start | N={n_way} K={k_shot} Q={q_query} | epochs={epochs} | episodes/epoch={episodes_per_epoch}")
    for ep in range(1, epochs+1):
        # ----- TRAIN (episodic xent) -----
        encoder.train()
        loader_tr = make_episode_loader(train_set, n_way, k_shot, q_query, episodes_per_epoch)
        loss_sum = 0.0; n_ep = 0
        for xs, _ in loader_tr:
            xs = xs.to(device)
            opt.zero_grad()
            emb = encoder(xs)
            P, Q, yq = split_support_query(emb, n_way, k_shot, q_query, device)
            if variant=='eucl':
                logits = logits_proto_euclidean(Q, P, tau=tau)
                loss = F.cross_entropy(logits, yq)
            else:
                logits = logits_proto_cosine(Q, P, alpha=alpha, tau=tau)
                loss = F.cross_entropy(logits, yq)
            loss.backward(); opt.step()
            loss_sum += loss.item(); n_ep += 1
        train_loss = loss_sum / max(1,n_ep)

        # ----- VALIDATION -----
        val_t1,val_t5 = eval_episodic_topk_proto(encoder, val_set, n_way, k_shot, q_query,
                                                 episodes=40, variant=variant, alpha=alpha, tau=tau)
        # ----- TEST (chỉ log) -----
        if log_test_each_epoch:
            test_t1,test_t5 = eval_episodic_topk_proto(encoder, test_set, n_way, k_shot, q_query,
                                                       episodes=40, variant=variant, alpha=alpha, tau=tau)
        else:
            test_t1=test_t5=float('nan')

        # ----- chọn best theo Val -----
        cur = val_t1 if best_by=='val@1' else val_t5
        is_best = (best_score is None) or (cur > best_score)
        if is_best:
            best_score = cur
            torch.save({"model_state": encoder.state_dict(),
                        "meta":{"label":label,"variant":variant,"enc_dim":enc_dim,
                                "alpha":alpha,"tau":tau,"best_by":best_by,"best_score":float(best_score)}}, ckpt)

        # ----- log -----
        print(f"[Proto-{variant}] Ep {ep:03d}/{epochs} | TrainLoss {train_loss:.4f} || "
              f"Val@1 {val_t1:.2f} Val@5 {val_t5:.2f} || Test@1 {test_t1:.2f} Test@5 {test_t5:.2f} || "
              f"Best({best_by}) {best_score:.2f} {'*' if is_best else ''}")
        write_csv_row(log_csv, [ep, round(train_loss,6), round(val_t1,4), round(val_t5,4),
                                (round(test_t1,4) if log_test_each_epoch else ""),
                                (round(test_t5,4) if log_test_each_epoch else ""),
                                best_by, int(is_best)])

    # ----- Final: TEST @ best checkpoint -----
    state = torch.load(ckpt, map_location='cpu')["model_state"]
    encoder.load_state_dict(state)
    t1,t5 = eval_episodic_topk_proto(encoder, test_set, n_way, k_shot, q_query, episodes=200,
                                     variant=variant, alpha=alpha, tau=tau)
    print(f"[Proto-{variant}] FINAL Test@1 {t1:.2f} | Test@5 {t5:.2f} | ckpt={ckpt}")
    return str(ckpt)

# === Huấn luyện 2 biến thể, lưu đúng tên file yêu cầu ===
EPOCHS = 20
EPISODES_PER_EPOCH = 200
N_WAY, K_SHOT, Q_QUERY = 5, 5, 15

ckpt_proto_e = train_prototypical(
    variant='eucl', n_way=N_WAY, k_shot=K_SHOT, q_query=Q_QUERY,
    episodes_per_epoch=EPISODES_PER_EPOCH, epochs=EPOCHS, enc_dim=64,
    alpha=10.0, tau=1.0, best_by='val@1', log_test_each_epoch=True
)

ckpt_proto_c = train_prototypical(
    variant='cos',  n_way=N_WAY, k_shot=K_SHOT, q_query=Q_QUERY,
    episodes_per_epoch=EPISODES_PER_EPOCH, epochs=EPOCHS, enc_dim=64,
    alpha=10.0, tau=1.0, best_by='val@1', log_test_each_epoch=True
)


[Proto-eucl] start | N=5 K=5 Q=15 | epochs=20 | episodes/epoch=200
[Proto-eucl] Ep 001/20 | TrainLoss 1.1177 || Val@1 57.67 Val@5 100.00 || Test@1 59.60 Test@5 100.00 || Best(val@1) 57.67 *
[Proto-eucl] Ep 002/20 | TrainLoss 0.9545 || Val@1 61.40 Val@5 100.00 || Test@1 64.60 Test@5 100.00 || Best(val@1) 61.40 *
[Proto-eucl] Ep 003/20 | TrainLoss 0.8690 || Val@1 67.17 Val@5 100.00 || Test@1 66.63 Test@5 100.00 || Best(val@1) 67.17 *
[Proto-eucl] Ep 004/20 | TrainLoss 0.7990 || Val@1 70.63 Val@5 100.00 || Test@1 69.87 Test@5 100.00 || Best(val@1) 70.63 *
[Proto-eucl] Ep 005/20 | TrainLoss 0.7240 || Val@1 70.90 Val@5 100.00 || Test@1 73.53 Test@5 100.00 || Best(val@1) 70.90 *
[Proto-eucl] Ep 006/20 | TrainLoss 0.6975 || Val@1 72.90 Val@5 100.00 || Test@1 72.30 Test@5 100.00 || Best(val@1) 72.90 *
[Proto-eucl] Ep 007/20 | TrainLoss 0.6452 || Val@1 76.13 Val@5 100.00 || Test@1 76.13 Test@5 100.00 || Best(val@1) 76.13 *
[Proto-eucl] Ep 008/20 | TrainLoss 0.6182 || Val@1 75.23 Val@5 100.00 ||

In [3]:
# ========= FEWSHOT PROTO: EVAL-ONLY (độc lập) =========
import torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader, Sampler
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN = (0.4914, 0.4822, 0.4465); STD=(0.2470, 0.2435, 0.2616)
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

test_set = datasets.CIFAR10("./data", train=False, download=True, transform=tf)
train_full = datasets.CIFAR10("./data", train=True, download=True, transform=tf)
torch.manual_seed(0)
perm = torch.randperm(len(train_full))
val_idx = perm[:5000]
val_set = Subset(train_full, val_idx.tolist())

class Conv4(nn.Module):
    def __init__(self, out_dim=64):
        super().__init__()
        def block(i,o): 
            return nn.Sequential(nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(True), nn.MaxPool2d(2))
        self.enc = nn.Sequential(block(3,32), block(32,64), block(64,64), block(64,out_dim))
        self.out_dim = out_dim
    def forward(self,x): return self.enc(x).mean(dim=[2,3])

class EpisodeSampler(Sampler):
    def __init__(self, base_set, n_way=5, k_shot=5, q_query=15, episodes_per_epoch=100):
        if isinstance(base_set, Subset):
            targets_full = np.array(base_set.dataset.targets)
            targets = targets_full[base_set.indices]
        else:
            targets = np.array(getattr(base_set, 'targets'))
        self.base = base_set
        self.targets = targets
        self.n_way, self.k_shot, self.q_query = n_way, k_shot, q_query
        self.episodes_per_epoch = episodes_per_epoch
        self.class2idx = {c: np.where(self.targets==c)[0] for c in np.unique(self.targets)}
    def __len__(self): return self.episodes_per_epoch * self.n_way * (self.k_shot + self.q_query)
    def __iter__(self):
        for _ in range(self.episodes_per_epoch):
            classes = np.random.choice(list(self.class2idx.keys()), self.n_way, replace=False)
            ep_idx=[]
            for c in classes:
                pool = self.class2idx[c]
                if len(pool) < self.k_shot + self.q_query:
                    pool = np.random.choice(pool, self.k_shot + self.q_query, replace=True)
                else:
                    pool = np.random.permutation(pool)[:self.k_shot + self.q_query]
                ep_idx += pool.tolist()
            yield from ep_idx

def make_episode_loader(base_set, n_way, k_shot, q_query, episodes, num_workers=2):
    sampler = EpisodeSampler(base_set, n_way=n_way, k_shot=k_shot, q_query=q_query, episodes_per_epoch=episodes)
    bs = n_way*(k_shot+q_query)
    return DataLoader(base_set, batch_size=bs, sampler=sampler, num_workers=num_workers, pin_memory=True)

def split_support_query(emb, n_way, k_shot, q_query, device):
    protos=[]; queries=[]; yq=[]
    for c in range(n_way):
        s = c*(k_shot+q_query)
        support = emb[s:s+k_shot]
        query   = emb[s+k_shot:s+k_shot+q_query]
        protos.append(support.mean(0, keepdim=True))
        queries.append(query)
        yq.append(torch.full((q_query,), c, device=device, dtype=torch.long))
    return torch.cat(protos,0), torch.cat(queries,0), torch.cat(yq,0)

def logits_proto_euclidean(Q, P, tau=1.0):
    d = torch.cdist(Q, P); return -d/tau
def logits_proto_cosine(Q, P, alpha=10.0, tau=1.0):
    q = F.normalize(Q, dim=1); p = F.normalize(P, dim=1); return (alpha*(q@p.t()))/tau

@torch.no_grad()
def eval_episodic_topk_proto(encoder, base_set, n_way, k_shot, q_query, episodes, variant='eucl', alpha=10.0, tau=1.0):
    encoder = encoder.to(device).eval()
    loader = make_episode_loader(base_set, n_way, k_shot, q_query, episodes)
    accs1=[]; accs5=[]
    topk = min(5, n_way)
    for xs,_ in loader:
        xs = xs.to(device)
        emb = encoder(xs)
        P,Q,yq = split_support_query(emb, n_way, k_shot, q_query, device)
        logits = logits_proto_euclidean(Q,P,tau) if variant=='eucl' else logits_proto_cosine(Q,P,alpha,tau)
        _, pred = logits.topk(topk,1,True,True)
        correct = pred.eq(yq.view(-1,1))
        accs1.append(100.0*correct[:, :1].sum().item()/yq.size(0))
        accs5.append(100.0*correct[:, :topk].sum().item()/yq.size(0))
    return float(np.mean(accs1)), float(np.mean(accs5))

def fsl_proto_eval_from_ckpt(ckpt_path:str, n_way=5, k_shot=5, q_query=15, episodes=200):
    payload = torch.load(ckpt_path, map_location='cpu')
    meta = payload.get('meta', {})
    enc_dim = meta.get('enc_dim', 64)
    variant = meta.get('variant', 'eucl' if 'eucl' in ckpt_path else ('cos' if 'cos' in ckpt_path else 'eucl'))
    alpha = meta.get('alpha', 10.0); tau = meta.get('tau', 1.0)

    encoder = Conv4(out_dim=enc_dim).to(device)
    encoder.load_state_dict(payload['model_state'], strict=False)

    val_t1,val_t5 = eval_episodic_topk_proto(encoder, val_set,  n_way, k_shot, q_query, episodes=100, variant=variant, alpha=alpha, tau=tau)
    tst_t1,tst_t5 = eval_episodic_topk_proto(encoder, test_set, n_way, k_shot, q_query, episodes=episodes,   variant=variant, alpha=alpha, tau=tau)
    print(f"[EVAL] {Path(ckpt_path).name} | Variant={variant} | "
          f"Val Top1 {val_t1:.2f} Top5 {val_t5:.2f} || Test Top1 {tst_t1:.2f} Top5 {tst_t5:.2f}")
    return (val_t1,val_t5),(tst_t1,tst_t5)

# ví dụ:
# fsl_proto_eval_from_ckpt("checkpoints/fsl_prototypical_networks_eucl.pth", n_way=5, k_shot=5, q_query=15, episodes=200)
# fsl_proto_eval_from_ckpt("checkpoints/fsl_prototypical_networks_cos.pth",  n_way=5, k_shot=5, q_query=15, episodes=200)


## (B) Siamese Networks

Cho đặc trưng \\( z=f_\theta(x) \\), khoảng cách \\( d(u,v) = \|u-v\|_2 \\).

### (B1) Contrastive Loss (học trên cặp)
Với cặp \\( (x_1,x_2) \\) và nhãn cặp \\( y\in\{0,1\} \\) (\\(1\\): cùng lớp, \\(0\\): khác lớp), margin \\( m>0 \\):
\\[
\mathcal{L}_{\text{contrast}}(\theta)
=\; y\; d\!\big(z_1,z_2\big)^2 \;+\; (1-y)\; \big[\,\max(0, m - d(z_1,z_2))\,\big]^2.
\\]

### (B2) Triplet Loss (học trên bộ 3)
Với bộ \\( (x^{\mathrm{a}}, x^{\mathrm{p}}, x^{\mathrm{n}}) \\) (anchor, positive cùng lớp; negative khác lớp):
\\[
\mathcal{L}_{\text{triplet}}(\theta)
=\; \max\!\big(0, \; d(z^{\mathrm{a}},z^{\mathrm{p}}) \;-\; d(z^{\mathrm{a}},z^{\mathrm{n}}) \;+\; m \big).
\\]

**Suy luận (inference):** phân loại bằng k-NN/cosine-NN trong không gian \\( f_\theta \\) hoặc huấn luyện **linear probe** nhỏ trên \\( f_\theta(x) \\).

**Đánh giá:** Top-1/Top-5 theo k-NN (val/test) hoặc theo probe.  
**Checkpoint:** `fsl_siamese_networks_contrastive.pth`, `fsl_siamese_networks_triplet.pth`.


In [5]:
# ========== SIAMESE: SHARED ==========
import os, csv, math, time, random, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, Sampler
from torchvision import datasets, transforms
from pathlib import Path

# --- device & folders ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKPT_DIR = Path("checkpoints"); CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR  = Path("logs"); LOG_DIR.mkdir(parents=True, exist_ok=True)

# --- CIFAR-10 split (fallback nếu bên ngoài chưa có) ---
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2470, 0.2435, 0.2616)
_tf  = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

if 'train_set' not in globals() or 'val_set' not in globals() or 'test_set' not in globals():
    _train_full = datasets.CIFAR10("./data", train=True,  download=True, transform=_tf)
    test_set    = datasets.CIFAR10("./data", train=False, download=True, transform=_tf)
    torch.manual_seed(0)
    _perm = torch.randperm(len(_train_full))
    _val_idx = _perm[:5000]; _tr_idx = _perm[5000:]
    train_set = Subset(_train_full, _tr_idx.tolist())
    val_set   = Subset(_train_full, _val_idx.tolist())

NUM_CLASSES = 10

# --- Encoder f_θ(x) → z ∈ R^d (gọn nhẹ) ---
class Conv4(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        def block(i,o):
            return nn.Sequential(
                nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(True), nn.MaxPool2d(2)
            )
        self.enc = nn.Sequential(block(3,32), block(32,64), block(64,64), block(64,out_dim))
        self.out_dim = out_dim
    def forward(self, x):          # [B,3,32,32] → [B,d]
        return self.enc(x).mean(dim=[2,3])

# --- Pair dataset cho Contrastive ---
class PairDataset(Dataset):
    def __init__(self, base_set):
        if isinstance(base_set, Subset):
            self.base = base_set
            targets = np.array(base_set.dataset.targets)[base_set.indices]
        else:
            self.base = base_set
            targets = np.array(getattr(base_set, 'targets'))
        self.targets = targets
        self.class2idx = {c: np.where(self.targets==c)[0] for c in np.unique(self.targets)}
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        x1, y1 = self.base[idx]
        if random.random() < 0.5:  # positive pair (y=1)
            j = int(np.random.choice(self.class2idx[y1])); y = 1
        else:                       # negative pair (y=0)
            negc = int(np.random.choice([c for c in self.class2idx if c!=y1]))
            j = int(np.random.choice(self.class2idx[negc])); y = 0
        x2, _ = self.base[j]
        return (x1, x2), y

# --- Triplet dataset cho Triplet loss ---
class TripletDataset(Dataset):
    def __init__(self, base_set):
        if isinstance(base_set, Subset):
            self.base = base_set
            targets = np.array(base_set.dataset.targets)[base_set.indices]
        else:
            self.base = base_set
            targets = np.array(getattr(base_set, 'targets'))
        self.targets = targets
        self.class2idx = {c: np.where(self.targets==c)[0] for c in np.unique(self.targets)}
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        a, ya = self.base[idx]
        p_idx = int(np.random.choice(self.class2idx[ya]))               # positive
        p, _  = self.base[p_idx]
        negc  = int(np.random.choice([c for c in self.class2idx if c!=ya]))
        n_idx = int(np.random.choice(self.class2idx[negc]))
        n, _  = self.base[n_idx]
        return (a, p, n), (ya, ya, negc)

# --- Episodic sampler N-way K-shot + Q query/way (đánh giá) ---
class EpisodeSampler(Sampler):
    def __init__(self, base_set, n_way=5, k_shot=5, q_query=15, episodes_per_epoch=100):
        if isinstance(base_set, Subset):
            targets_full = np.array(base_set.dataset.targets)
            targets = targets_full[base_set.indices]
        else:
            targets = np.array(getattr(base_set, 'targets'))
        self.base = base_set
        self.targets = targets
        self.n_way, self.k_shot, self.q_query = n_way, k_shot, q_query
        self.episodes_per_epoch = episodes_per_epoch
        self.class2idx = {c: np.where(self.targets==c)[0] for c in np.unique(self.targets)}
    def __len__(self): return self.episodes_per_epoch * self.n_way * (self.k_shot + self.q_query)
    def __iter__(self):
        for _ in range(self.episodes_per_epoch):
            classes = np.random.choice(list(self.class2idx.keys()), self.n_way, replace=False)
            ep_idx=[]
            for c in classes:
                pool = self.class2idx[c]
                if len(pool) < self.k_shot + self.q_query:
                    pool = np.random.choice(pool, self.k_shot + self.q_query, replace=True)
                else:
                    pool = np.random.permutation(pool)[:self.k_shot + self.q_query]
                ep_idx += pool.tolist()
            yield from ep_idx

def make_episode_loader(base_set, n_way, k_shot, q_query, episodes, num_workers=2):
    sampler = EpisodeSampler(base_set, n_way=n_way, k_shot=k_shot, q_query=q_query, episodes_per_epoch=episodes)
    bs = n_way*(k_shot+q_query)
    return DataLoader(base_set, batch_size=bs, sampler=sampler, num_workers=num_workers, pin_memory=True)

# --- Ký hiệu & hàm trợ giúp ---
def write_csv_header(path, header):
    with open(path, "w", newline="") as f: csv.writer(f).writerow(header)
def write_csv_row(path, row):
    with open(path, "a", newline="") as f: csv.writer(f).writerow(row)

# --- ĐÁNH GIÁ: k-NN/cosine-NN trên episodic episodes ---
@torch.no_grad()
def eval_episodic_topk_knn(encoder, base_set, n_way, k_shot, q_query, episodes, metric='cosine'):
    """
    metric: 'cosine' (mặc định) | 'euclidean'
    - Dự đoán Top-k theo điểm lớp = max_{mẫu support thuộc lớp c} sim(q, s)
      (tương đương 1-NN theo mẫu; để lấy Top-5 theo lớp).
    """
    encoder = encoder.to(device).eval()
    loader = make_episode_loader(base_set, n_way, k_shot, q_query, episodes)
    topk = min(5, n_way)
    acc1, acc5 = [], []

    for xs, _ in loader:
        xs = xs.to(device)
        z = encoder(xs)                             # [N(K+Q), d]
        s_feats=[]; s_lbl=[]
        q_feats=[]; q_lbl=[]
        for c in range(n_way):
            s = c*(k_shot+q_query)
            s_idx = slice(s, s+k_shot)
            q_idx = slice(s+k_shot, s+k_shot+q_query)
            s_feats.append(z[s_idx]); s_lbl.append(torch.full((k_shot,), c, device=device, dtype=torch.long))
            q_feats.append(z[q_idx]); q_lbl.append(torch.full((q_query,), c, device=device, dtype=torch.long))
        S = torch.cat(s_feats,0)    # [NK, d]
        yS= torch.cat(s_lbl,0)      # [NK]
        Q = torch.cat(q_feats,0)    # [NQ, d]
        yQ= torch.cat(q_lbl,0)      # [NQ]

        if metric=='cosine':
            S = F.normalize(S, dim=1); Q = F.normalize(Q, dim=1)
            score = Q @ S.t()       # [NQ, NK] (càng lớn càng tốt)
        else:
            # dùng -||q-s|| để có "điểm" càng lớn càng tốt
            score = -torch.cdist(Q, S)  # [NQ, NK]

        # 1-NN theo mẫu: lớp dự đoán = nhãn support có score lớn nhất
        nn_idx = score.argmax(dim=1)         # [NQ]
        pred1  = yS[nn_idx]                  # [NQ]
        top1 = (pred1==yQ).float().mean().item()*100.0

        # Top-5 theo lớp: dùng "max theo lớp"
        # class_scores[c] = max score trên các mẫu support thuộc lớp c
        class_scores = torch.stack([
            score[:, (yS==c).nonzero(as_tuple=True)[0]].max(dim=1).values
            for c in range(n_way)
        ], dim=1)                            # [NQ, N]
        _, pred_topk = class_scores.topk(topk, dim=1, largest=True, sorted=True)
        correct_topk = (pred_topk==yQ.view(-1,1)).any(dim=1).float().mean().item()*100.0

        acc1.append(top1); acc5.append(correct_topk)
    return float(np.mean(acc1)), float(np.mean(acc5))


In [6]:
# ========== SIAMESE: CONTRASTIVE TRAIN ==========
def contrastive_loss(z1, z2, y, margin=1.0):
    # y ∈ {0,1}; d = ||z1 - z2||_2
    d = F.pairwise_distance(z1, z2)
    return (y*(d**2) + (1-y)*F.relu(margin - d)**2).mean()

def train_siamese_contrastive(
    n_way=5, k_shot=5, q_query=15,
    epochs=20, enc_dim=128, batch=256, margin=1.0,
    eval_metric='cosine',       # 'cosine' | 'euclidean' (k-NN)
    episodes_eval=40,
    label='fsl_siamese_networks_contrastive',
    best_by='val@1', log_test_each_epoch=True
):
    encoder = Conv4(out_dim=enc_dim).to(device)
    opt = torch.optim.Adam(encoder.parameters(), lr=1e-3)
    dl = DataLoader(PairDataset(train_set), batch_size=batch, shuffle=True, num_workers=2)

    log_csv = LOG_DIR/f"{label}.csv"
    write_csv_header(log_csv, ["epoch","train_loss","val_top1","val_top5","test_top1","test_top5","best_by","is_best"])

    best_score=None; ckpt=CKPT_DIR/f"{label}.pth"
    print(f"[Siamese-Contrast] start | epochs={epochs} | margin={margin} | eval_metric={eval_metric}")

    for ep in range(1, epochs+1):
        # ----- TRAIN -----
        encoder.train(); losses=[]
        for (x1,x2), y in dl:
            x1,x2 = x1.to(device), x2.to(device)
            y = torch.tensor(y, dtype=torch.float32, device=device)
            opt.zero_grad()
            z1,z2 = encoder(x1), encoder(x2)
            loss = contrastive_loss(z1,z2,y,margin=margin)
            loss.backward(); opt.step()
            losses.append(loss.item())
        train_loss = float(np.mean(losses)) if losses else 0.0

        # ----- VAL/TEST episodic k-NN -----
        val_t1,val_t5 = eval_episodic_topk_knn(encoder, val_set,  n_way, k_shot, q_query, episodes=episodes_eval, metric=eval_metric)
        if log_test_each_epoch:
            test_t1,test_t5 = eval_episodic_topk_knn(encoder, test_set, n_way, k_shot, q_query, episodes=episodes_eval, metric=eval_metric)
        else:
            test_t1=test_t5=float('nan')

        # ----- chọn best -----
        cur = val_t1 if best_by=='val@1' else val_t5
        is_best = (best_score is None) or (cur>best_score)
        if is_best:
            best_score=cur
            torch.save({"model_state": encoder.state_dict(),
                        "meta":{"label":label,"enc_dim":enc_dim,"margin":margin,
                                "eval_metric":eval_metric,"best_by":best_by,
                                "best_score":float(best_score)}}, ckpt)

        # ----- log -----
        print(f"[Siamese-Contrast] Ep {ep:03d}/{epochs} | TrainLoss {train_loss:.4f} || "
              f"Val@1 {val_t1:.2f} Val@5 {val_t5:.2f} || Test@1 {test_t1:.2f} Test@5 {test_t5:.2f} || "
              f"Best({best_by}) {best_score:.2f} {'*' if is_best else ''}")
        write_csv_row(log_csv, [ep, round(train_loss,6), round(val_t1,4), round(val_t5,4),
                                (round(test_t1,4) if log_test_each_epoch else ""),
                                (round(test_t5,4) if log_test_each_epoch else ""),
                                best_by, int(is_best)])

    # ----- Final: TEST @ best -----
    state = torch.load(ckpt, map_location='cpu')["model_state"]
    encoder.load_state_dict(state)
    t1,t5 = eval_episodic_topk_knn(encoder, test_set, n_way, k_shot, q_query, episodes=200, metric=eval_metric)
    print(f"[Siamese-Contrast] FINAL Test@1 {t1:.2f} | Test@5 {t5:.2f} | ckpt={ckpt}")
    return str(ckpt)

# === Train Contrastive ===
ckpt_siam_c = train_siamese_contrastive(
    n_way=5, k_shot=5, q_query=15, epochs=20, enc_dim=128, batch=256, margin=1.0,
    eval_metric='cosine', episodes_eval=40,
    label='fsl_siamese_networks_contrastive', best_by='val@1', log_test_each_epoch=True
)


[Siamese-Contrast] start | epochs=20 | margin=1.0 | eval_metric=cosine


/tmp/ipykernel_36/2004891965.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y, dtype=torch.float32, device=device)


[Siamese-Contrast] Ep 001/20 | TrainLoss 0.7549 || Val@1 35.93 Val@5 100.00 || Test@1 38.03 Test@5 100.00 || Best(val@1) 35.93 *
[Siamese-Contrast] Ep 002/20 | TrainLoss 0.2251 || Val@1 41.63 Val@5 100.00 || Test@1 39.87 Test@5 100.00 || Best(val@1) 41.63 *
[Siamese-Contrast] Ep 003/20 | TrainLoss 0.2147 || Val@1 43.93 Val@5 100.00 || Test@1 42.43 Test@5 100.00 || Best(val@1) 43.93 *
[Siamese-Contrast] Ep 004/20 | TrainLoss 0.2074 || Val@1 45.17 Val@5 100.00 || Test@1 46.77 Test@5 100.00 || Best(val@1) 45.17 *
[Siamese-Contrast] Ep 005/20 | TrainLoss 0.2042 || Val@1 46.97 Val@5 100.00 || Test@1 48.50 Test@5 100.00 || Best(val@1) 46.97 *
[Siamese-Contrast] Ep 006/20 | TrainLoss 0.1993 || Val@1 50.87 Val@5 100.00 || Test@1 50.93 Test@5 100.00 || Best(val@1) 50.87 *
[Siamese-Contrast] Ep 007/20 | TrainLoss 0.1967 || Val@1 51.47 Val@5 100.00 || Test@1 51.83 Test@5 100.00 || Best(val@1) 51.47 *
[Siamese-Contrast] Ep 008/20 | TrainLoss 0.1926 || Val@1 51.20 Val@5 100.00 || Test@1 53.17 Test@

In [7]:
# ========== SIAMESE: TRIPLET TRAIN ==========
def train_siamese_triplet(
    n_way=5, k_shot=5, q_query=15,
    epochs=20, enc_dim=128, batch=256, margin=0.5,
    eval_metric='cosine',     # 'cosine' | 'euclidean'
    episodes_eval=40,
    label='fsl_siamese_networks_triplet',
    best_by='val@1', log_test_each_epoch=True
):
    encoder = Conv4(out_dim=enc_dim).to(device)
    opt = torch.optim.Adam(encoder.parameters(), lr=1e-3)
    criterion = nn.TripletMarginLoss(margin=margin, p=2)  # theo công thức: max(0, d(ap)-d(an)+m)
    dl = DataLoader(TripletDataset(train_set), batch_size=batch, shuffle=True, num_workers=2)

    log_csv = LOG_DIR/f"{label}.csv"
    write_csv_header(log_csv, ["epoch","train_loss","val_top1","val_top5","test_top1","test_top5","best_by","is_best"])

    best_score=None; ckpt=CKPT_DIR/f"{label}.pth"
    print(f"[Siamese-Triplet] start | epochs={epochs} | margin={margin} | eval_metric={eval_metric}")

    for ep in range(1, epochs+1):
        # ----- TRAIN -----
        encoder.train(); losses=[]
        for (xa,xp,xn), _ in dl:
            xa,xp,xn = xa.to(device), xp.to(device), xn.to(device)
            opt.zero_grad()
            za,zp,zn = encoder(xa), encoder(xp), encoder(xn)
            loss = criterion(za,zp,zn)
            loss.backward(); opt.step()
            losses.append(loss.item())
        train_loss = float(np.mean(losses)) if losses else 0.0

        # ----- VAL/TEST episodic k-NN -----
        val_t1,val_t5 = eval_episodic_topk_knn(encoder, val_set,  n_way, k_shot, q_query, episodes=episodes_eval, metric=eval_metric)
        if log_test_each_epoch:
            test_t1,test_t5 = eval_episodic_topk_knn(encoder, test_set, n_way, k_shot, q_query, episodes=episodes_eval, metric=eval_metric)
        else:
            test_t1=test_t5=float('nan')

        # ----- chọn best -----
        cur = val_t1 if best_by=='val@1' else val_t5
        is_best = (best_score is None) or (cur>best_score)
        if is_best:
            best_score=cur
            torch.save({"model_state": encoder.state_dict(),
                        "meta":{"label":label,"enc_dim":enc_dim,"margin":margin,
                                "eval_metric":eval_metric,"best_by":best_by,
                                "best_score":float(best_score)}}, ckpt)

        # ----- log -----
        print(f"[Siamese-Triplet] Ep {ep:03d}/{epochs} | TrainLoss {train_loss:.4f} || "
              f"Val@1 {val_t1:.2f} Val@5 {val_t5:.2f} || Test@1 {test_t1:.2f} Test@5 {test_t5:.2f} || "
              f"Best({best_by}) {best_score:.2f} {'*' if is_best else ''}")
        write_csv_row(log_csv, [ep, round(train_loss,6), round(val_t1,4), round(val_t5,4),
                                (round(test_t1,4) if log_test_each_epoch else ""),
                                (round(test_t5,4) if log_test_each_epoch else ""),
                                best_by, int(is_best)])

    # ----- Final: TEST @ best -----
    state = torch.load(ckpt, map_location='cpu')["model_state"]
    encoder.load_state_dict(state)
    t1,t5 = eval_episodic_topk_knn(encoder, test_set, n_way, k_shot, q_query, episodes=200, metric=eval_metric)
    print(f"[Siamese-Triplet] FINAL Test@1 {t1:.2f} | Test@5 {t5:.2f} | ckpt={ckpt}")
    return str(ckpt)

# === Train Triplet ===
ckpt_siam_t = train_siamese_triplet(
    n_way=5, k_shot=5, q_query=15, epochs=20, enc_dim=128, batch=256, margin=0.5,
    eval_metric='cosine', episodes_eval=40,
    label='fsl_siamese_networks_triplet', best_by='val@1', log_test_each_epoch=True
)


[Siamese-Triplet] start | epochs=20 | margin=0.5 | eval_metric=cosine
[Siamese-Triplet] Ep 001/20 | TrainLoss 0.3289 || Val@1 52.93 Val@5 100.00 || Test@1 51.97 Test@5 100.00 || Best(val@1) 52.93 *
[Siamese-Triplet] Ep 002/20 | TrainLoss 0.2531 || Val@1 58.40 Val@5 100.00 || Test@1 56.63 Test@5 100.00 || Best(val@1) 58.40 *
[Siamese-Triplet] Ep 003/20 | TrainLoss 0.2296 || Val@1 59.93 Val@5 100.00 || Test@1 57.10 Test@5 100.00 || Best(val@1) 59.93 *
[Siamese-Triplet] Ep 004/20 | TrainLoss 0.2126 || Val@1 60.40 Val@5 100.00 || Test@1 60.83 Test@5 100.00 || Best(val@1) 60.40 *
[Siamese-Triplet] Ep 005/20 | TrainLoss 0.1992 || Val@1 65.57 Val@5 100.00 || Test@1 61.83 Test@5 100.00 || Best(val@1) 65.57 *
[Siamese-Triplet] Ep 006/20 | TrainLoss 0.1933 || Val@1 65.27 Val@5 100.00 || Test@1 65.40 Test@5 100.00 || Best(val@1) 65.57 
[Siamese-Triplet] Ep 007/20 | TrainLoss 0.1829 || Val@1 66.07 Val@5 100.00 || Test@1 64.43 Test@5 100.00 || Best(val@1) 66.07 *
[Siamese-Triplet] Ep 008/20 | Train

In [8]:
# ========== SIAMESE: EVAL-ONLY (độc lập) ==========
import torch, numpy as np, random
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, Sampler
from torchvision import datasets, transforms
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2470, 0.2435, 0.2616)
_tf  = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

# data
train_full = datasets.CIFAR10("./data", train=True,  download=True, transform=_tf)
test_set   = datasets.CIFAR10("./data", train=False, download=True, transform=_tf)
torch.manual_seed(0)
_perm = torch.randperm(len(train_full))
val_idx = _perm[:5000]
val_set = Subset(train_full, val_idx.tolist())

# encoder
class Conv4(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        def block(i,o):
            return nn.Sequential(nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(True), nn.MaxPool2d(2))
        self.enc = nn.Sequential(block(3,32), block(32,64), block(64,64), block(64,out_dim))
        self.out_dim = out_dim
    def forward(self, x): return self.enc(x).mean(dim=[2,3])

# episodic loader
class EpisodeSampler(Sampler):
    def __init__(self, base_set, n_way=5, k_shot=5, q_query=15, episodes_per_epoch=100):
        if isinstance(base_set, Subset):
            targets_full = np.array(base_set.dataset.targets)
            targets = targets_full[base_set.indices]
        else:
            targets = np.array(getattr(base_set, 'targets'))
        self.base = base_set; self.targets = targets
        self.n_way, self.k_shot, self.q_query = n_way, k_shot, q_query
        self.episodes_per_epoch = episodes_per_epoch
        self.class2idx = {c: np.where(self.targets==c)[0] for c in np.unique(self.targets)}
    def __len__(self): return self.episodes_per_epoch * self.n_way * (self.k_shot + self.q_query)
    def __iter__(self):
        for _ in range(self.episodes_per_epoch):
            classes = np.random.choice(list(self.class2idx.keys()), self.n_way, replace=False)
            ep_idx=[]
            for c in classes:
                pool = self.class2idx[c]
                if len(pool) < self.k_shot + self.q_query:
                    pool = np.random.choice(pool, self.k_shot + self.q_query, replace=True)
                else:
                    pool = np.random.permutation(pool)[:self.k_shot + self.q_query]
                ep_idx += pool.tolist()
            yield from ep_idx

def make_episode_loader(base_set, n_way, k_shot, q_query, episodes, num_workers=2):
    sampler = EpisodeSampler(base_set, n_way=n_way, k_shot=k_shot, q_query=q_query, episodes_per_epoch=episodes)
    bs = n_way*(k_shot+q_query)
    return DataLoader(base_set, batch_size=bs, sampler=sampler, num_workers=num_workers, pin_memory=True)

@torch.no_grad()
def eval_episodic_topk_knn(encoder, base_set, n_way, k_shot, q_query, episodes, metric='cosine'):
    encoder = encoder.to(device).eval()
    loader = make_episode_loader(base_set, n_way, k_shot, q_query, episodes)
    topk = min(5, n_way)
    acc1, acc5 = [], []
    for xs,_ in loader:
        xs = xs.to(device)
        z = encoder(xs)
        s_feats=[]; s_lbl=[]; q_feats=[]; q_lbl=[]
        for c in range(n_way):
            s = c*(k_shot+q_query)
            s_idx = slice(s, s+k_shot)
            q_idx = slice(s+k_shot, s+k_shot+q_query)
            s_feats.append(z[s_idx]); s_lbl.append(torch.full((k_shot,), c, device=device, dtype=torch.long))
            q_feats.append(z[q_idx]); q_lbl.append(torch.full((q_query,), c, device=device, dtype=torch.long))
        S = torch.cat(s_feats,0); yS= torch.cat(s_lbl,0)
        Q = torch.cat(q_feats,0); yQ= torch.cat(q_lbl,0)

        if metric=='cosine':
            S = F.normalize(S, dim=1); Q = F.normalize(Q, dim=1)
            score = Q @ S.t()
        else:
            score = -torch.cdist(Q, S)

        nn_idx = score.argmax(dim=1)
        pred1 = yS[nn_idx]
        top1 = (pred1==yQ).float().mean().item()*100.0

        class_scores = torch.stack([
            score[:, (yS==c).nonzero(as_tuple=True)[0]].max(dim=1).values
            for c in range(n_way)
        ], dim=1)
        _, pred_topk = class_scores.topk(topk, dim=1, largest=True, sorted=True)
        top5 = (pred_topk==yQ.view(-1,1)).any(dim=1).float().mean().item()*100.0

        acc1.append(top1); acc5.append(top5)
    return float(np.mean(acc1)), float(np.mean(acc5))

def siamese_eval_from_ckpt(ckpt_path:str, n_way=5, k_shot=5, q_query=15, episodes=200, metric='cosine'):
    payload = torch.load(ckpt_path, map_location='cpu')
    meta = payload.get('meta', {})
    enc_dim = meta.get('enc_dim', 128)

    enc = Conv4(out_dim=enc_dim).to(device)
    enc.load_state_dict(payload['model_state'], strict=False)

    val_t1,val_t5 = eval_episodic_topk_knn(enc, val_set,  n_way, k_shot, q_query, episodes=100, metric=metric)
    tst_t1,tst_t5 = eval_episodic_topk_knn(enc, test_set, n_way, k_shot, q_query, episodes=episodes, metric=metric)
    print(f"[EVAL] {Path(ckpt_path).name} | metric={metric} | "
          f"Val Top1 {val_t1:.2f} Top5 {val_t5:.2f} || Test Top1 {tst_t1:.2f} Top5 {tst_t5:.2f}")
    return (val_t1,val_t5),(tst_t1,tst_t5)

# ví dụ:
# siamese_eval_from_ckpt("checkpoints/fsl_siamese_networks_contrastive.pth", n_way=5, k_shot=5, q_query=15, episodes=200, metric='cosine')
# siamese_eval_from_ckpt("checkpoints/fsl_siamese_networks_triplet.pth",    n_way=5, k_shot=5, q_query=15, episodes=200, metric='cosine')


## (C) Matching Networks (Cosine Attention)

Với **support** \\( \mathcal{S}=\{(x_i^{\mathrm{s}}, y_i^{\mathrm{s}})\}_{i=1}^{NK} \\),
đặc trưng \\( s_i=f_\theta(x_i^{\mathrm{s}}) \\) và **query** \\( q=f_\theta(x^{\mathrm{q}}) \\).

**Attention theo cosine** cho từng phần tử support:
\\[
a_i(q) \;=\; \frac{\exp\!\left(\alpha\, s_{\cos}\!\left(q,\, s_i\right)\right)}
{\sum\limits_{j=1}^{NK}\exp\!\left(\alpha\, s_{\cos}\!\left(q,\, s_j\right)\right)}.
\\]

**Phân phối dự đoán của query** (gộp theo nhãn support):
\\[
p_\theta\!\left(y=c\mid x^{\mathrm{q}}\right)
\;=\; \sum_{i=1}^{NK} a_i(q)\; \mathbf{1}[\,y_i^{\mathrm{s}}=c\,].
\\]

**Hàm mất mát (negative log-likelihood) theo tập**:
\\[
\mathcal{L}_{\text{match}}(\theta)
\;=\; - \frac{1}{NQ} \sum_{c=1}^N \sum_{j=1}^Q 
\log p_\theta\!\left(y=c \mid x_{c,j}^{\mathrm{q}}\right).
\\]

**Ghi chú:** Biến thể ở đây **không** dùng FCE/LSTM để giữ mô hình **nhẹ**; chỉ dùng attention cosine.  
**Checkpoint:** `fsl_matching_networks_cosine.pth`.


In [9]:
# ===== Matching Networks (Cosine Attention) — TRAIN =====
import os, csv, math, time, random, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, Sampler
from torchvision import datasets, transforms
from pathlib import Path

# ----- device & folders -----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKPT_DIR = Path("checkpoints"); CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR  = Path("logs"); LOG_DIR.mkdir(parents=True, exist_ok=True)

# ----- (re)use CIFAR10 splits if có; nếu chưa có thì tự tạo -----
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2470, 0.2435, 0.2616)
_tf  = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

if 'train_set' not in globals() or 'val_set' not in globals() or 'test_set' not in globals():
    _train_full = datasets.CIFAR10("./data", train=True,  download=True, transform=_tf)
    test_set    = datasets.CIFAR10("./data", train=False, download=True, transform=_tf)
    torch.manual_seed(0)
    _perm = torch.randperm(len(_train_full))
    _val_idx = _perm[:5000]; _tr_idx = _perm[5000:]
    train_set = Subset(_train_full, _tr_idx.tolist())
    val_set   = Subset(_train_full, _val_idx.tolist())

# ----- encoder gọn nhẹ f_θ -----
if 'Conv4' not in globals():
    class Conv4(nn.Module):
        def __init__(self, out_dim=64):
            super().__init__()
            def block(i,o):
                return nn.Sequential(
                    nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(True), nn.MaxPool2d(2)
                )
            self.enc = nn.Sequential(block(3,32), block(32,64), block(64,64), block(64,out_dim))
            self.out_dim = out_dim
        def forward(self, x):              # [B,3,32,32] → [B,d]
            return self.enc(x).mean(dim=[2,3])

# ----- Episodic sampler N-way K-shot + Q query/way -----
class EpisodeSampler(Sampler):
    def __init__(self, base_set, n_way=5, k_shot=5, q_query=15, episodes_per_epoch=100):
        if isinstance(base_set, Subset):
            targets_full = np.array(base_set.dataset.targets)
            targets = targets_full[base_set.indices]
        else:
            targets = np.array(getattr(base_set, 'targets'))
        self.base = base_set
        self.targets = targets
        self.n_way, self.k_shot, self.q_query = n_way, k_shot, q_query
        self.episodes_per_epoch = episodes_per_epoch
        self.class2idx = {c: np.where(self.targets==c)[0] for c in np.unique(self.targets)}
    def __len__(self):
        return self.episodes_per_epoch * self.n_way * (self.k_shot + self.q_query)
    def __iter__(self):
        for _ in range(self.episodes_per_epoch):
            classes = np.random.choice(list(self.class2idx.keys()), self.n_way, replace=False)
            ep_idx=[]
            for c in classes:
                pool = self.class2idx[c]
                if len(pool) < self.k_shot + self.q_query:
                    pool = np.random.choice(pool, self.k_shot + self.q_query, replace=True)
                else:
                    pool = np.random.permutation(pool)[:self.k_shot + self.q_query]
                ep_idx += pool.tolist()
            yield from ep_idx

def make_episode_loader(base_set, n_way, k_shot, q_query, episodes, num_workers=2):
    sampler = EpisodeSampler(base_set, n_way=n_way, k_shot=k_shot, q_query=q_query, episodes_per_epoch=episodes)
    bs = n_way*(k_shot+q_query)
    return DataLoader(base_set, batch_size=bs, sampler=sampler, num_workers=num_workers, pin_memory=True)

# ----- Matching Networks: logits/log-prob theo công thức -----
def logits_matching_cosine(q_feats, s_feats, s_labels, n_way, alpha=10.0):
    q = F.normalize(q_feats, dim=1); s = F.normalize(s_feats, dim=1)     # cosine sim
    sim = q @ s.t()                                                      # [NQ, NK]
    attn = F.softmax(alpha*sim, dim=1)                                   # a_i(q)
    onehot = F.one_hot(s_labels, num_classes=n_way).float()              # [NK, N]
    probs = attn @ onehot                                                # [NQ, N]
    return torch.log(torch.clamp(probs, 1e-9, 1.0))                      # log p(y|q)

# ----- Đánh giá episodic Top-1/Top-5 cho Matching -----
@torch.no_grad()
def eval_episodic_topk_matching(encoder, base_set, n_way, k_shot, q_query, episodes, alpha=10.0):
    encoder = encoder.to(device).eval()
    loader = make_episode_loader(base_set, n_way, k_shot, q_query, episodes)
    accs1=[]; accs5=[]
    topk = min(5, n_way)
    for xs, _ in loader:
        xs = xs.to(device)
        emb = encoder(xs)
        # tách S/Q
        s_feats=[]; s_lbl=[]; q_feats=[]; q_lbl=[]
        for c in range(n_way):
            s = c*(k_shot+q_query)
            s_idx = slice(s, s+k_shot); q_idx = slice(s+k_shot, s+k_shot+q_query)
            s_feats.append(emb[s_idx]); s_lbl.append(torch.full((k_shot,), c, device=device, dtype=torch.long))
            q_feats.append(emb[q_idx]); q_lbl.append(torch.full((q_query,), c, device=device, dtype=torch.long))
        s_feats = torch.cat(s_feats,0); s_lbl = torch.cat(s_lbl,0)
        q_feats = torch.cat(q_feats,0); q_lbl = torch.cat(q_lbl,0)

        logp = logits_matching_cosine(q_feats, s_feats, s_lbl, n_way=n_way, alpha=alpha)  # [NQ,N]
        _, pred = logp.topk(topk, 1, True, True)
        correct = pred.eq(q_lbl.view(-1,1))
        accs1.append(100.0*correct[:, :1].sum().item()/q_lbl.size(0))
        accs5.append(100.0*correct[:, :topk].sum().item()/q_lbl.size(0))
    return float(np.mean(accs1)), float(np.mean(accs5))

# ----- tiện ích log CSV -----
def write_csv_header(path, header):
    with open(path, "w", newline="") as f: csv.writer(f).writerow(header)
def write_csv_row(path, row):
    with open(path, "a", newline="") as f: csv.writer(f).writerow(row)

# ----- FLOPs/latency (tuỳ chọn) -----
def try_flops(model, input_size=(1,3,32,32)):
    try:
        from thop import profile
        dummy = torch.randn(*input_size).to(next(model.parameters()).device)
        macs,_ = profile(model, inputs=(dummy,), verbose=False)
        return int(macs*2)
    except Exception:
        return "N/A"

@torch.no_grad()
def benchmark_latency(model, input_size=(1,3,32,32), nwarm=10, niter=30):
    model.eval()
    x = torch.randn(*input_size).to(next(model.parameters()).device)
    for _ in range(nwarm): _ = model(x)
    ts=[]
    for _ in range(niter):
        t0=time.time(); _=model(x); ts.append(time.time()-t0)
    return 1000*np.mean(ts)

# ----- Huấn luyện Matching Networks (cosine attention) -----
def train_matching(
    n_way=5, k_shot=5, q_query=15,
    episodes_per_epoch=200, epochs=20, enc_dim=64, alpha=10.0,
    label='fsl_matching_networks_cosine', best_by='val@1',
    log_test_each_epoch=True
):
    encoder = Conv4(out_dim=enc_dim).to(device)
    opt = torch.optim.Adam(encoder.parameters(), lr=1e-3)

    log_csv = LOG_DIR/f"{label}.csv"
    write_csv_header(log_csv, ["epoch","train_loss","val_top1","val_top5","test_top1","test_top5","best_by","is_best"])

    best_score=None; ckpt=CKPT_DIR/f"{label}.pth"
    print(f"[Matching] start | N={n_way} K={k_shot} Q={q_query} | epochs={epochs} | episodes/epoch={episodes_per_epoch}")

    for ep in range(1, epochs+1):
        # ---- TRAIN: episodic NLL theo công thức ----
        encoder.train()
        loader_tr = make_episode_loader(train_set, n_way, k_shot, q_query, episodes_per_epoch)
        loss_sum = 0.0; n_ep = 0
        for xs, _ in loader_tr:
            xs = xs.to(device)
            opt.zero_grad()
            emb = encoder(xs)
            # tách S/Q
            s_feats=[]; s_lbl=[]; q_feats=[]; q_lbl=[]
            for c in range(n_way):
                s = c*(k_shot+q_query)
                s_idx = slice(s, s+k_shot); q_idx = slice(s+k_shot, s+k_shot+q_query)
                s_feats.append(emb[s_idx]); s_lbl.append(torch.full((k_shot,), c, device=device, dtype=torch.long))
                q_feats.append(emb[q_idx]); q_lbl.append(torch.full((q_query,), c, device=device, dtype=torch.long))
            s_feats = torch.cat(s_feats,0); s_lbl = torch.cat(s_lbl,0)
            q_feats = torch.cat(q_feats,0); q_lbl = torch.cat(q_lbl,0)

            logp = logits_matching_cosine(q_feats, s_feats, s_lbl, n_way=n_way, alpha=alpha)  # log-prob
            loss = F.nll_loss(logp, q_lbl)
            loss.backward(); opt.step()
            loss_sum += loss.item(); n_ep += 1
        train_loss = loss_sum / max(1,n_ep)

        # ---- VALIDATION / TEST ----
        val_t1,val_t5 = eval_episodic_topk_matching(encoder, val_set,  n_way, k_shot, q_query, episodes=40, alpha=alpha)
        if log_test_each_epoch:
            test_t1,test_t5 = eval_episodic_topk_matching(encoder, test_set, n_way, k_shot, q_query, episodes=40, alpha=alpha)
        else:
            test_t1=test_t5=float('nan')

        # ---- chọn best theo Val ----
        cur = val_t1 if best_by=='val@1' else val_t5
        is_best = (best_score is None) or (cur>best_score)
        if is_best:
            best_score=cur
            torch.save({"model_state": encoder.state_dict(),
                        "meta":{"label":label,"enc_dim":enc_dim,"alpha":alpha,
                                "best_by":best_by,"best_score":float(best_score)}}, ckpt)

        # ---- log ----
        print(f"[Matching] Ep {ep:03d}/{epochs} | TrainLoss {train_loss:.4f} || "
              f"Val@1 {val_t1:.2f} Val@5 {val_t5:.2f} || Test@1 {test_t1:.2f} Test@5 {test_t5:.2f} || "
              f"Best({best_by}) {best_score:.2f} {'*' if is_best else ''}")
        write_csv_row(log_csv, [ep, round(train_loss,6), round(val_t1,4), round(val_t5,4),
                                (round(test_t1,4) if log_test_each_epoch else ""),
                                (round(test_t5,4) if log_test_each_epoch else ""),
                                best_by, int(is_best)])

    # ---- Final: TEST @ best checkpoint ----
    state = torch.load(ckpt, map_location='cpu')["model_state"]
    encoder.load_state_dict(state)
    t1,t5 = eval_episodic_topk_matching(encoder, test_set, n_way, k_shot, q_query, episodes=200, alpha=alpha)
    params = sum(p.numel() for p in encoder.parameters())
    flops  = try_flops(encoder); lat = benchmark_latency(encoder)
    print(f"[Matching] FINAL Test@1 {t1:.2f} | Test@5 {t5:.2f} | Params:{params:,} | FLOPs:{flops} | Lat(ms):{lat:.2f} | ckpt={ckpt}")
    return str(ckpt)

# === Gọi train (giữ đúng tên checkpoint yêu cầu) ===
ckpt_match = train_matching(
    n_way=5, k_shot=5, q_query=15,
    episodes_per_epoch=200, epochs=20, enc_dim=64, alpha=10.0,
    label='fsl_matching_networks_cosine', best_by='val@1',
    log_test_each_epoch=True
)


[Matching] start | N=5 K=5 Q=15 | epochs=20 | episodes/epoch=200
[Matching] Ep 001/20 | TrainLoss 1.1510 || Val@1 56.00 Val@5 100.00 || Test@1 54.63 Test@5 100.00 || Best(val@1) 56.00 *
[Matching] Ep 002/20 | TrainLoss 0.9711 || Val@1 62.47 Val@5 100.00 || Test@1 62.40 Test@5 100.00 || Best(val@1) 62.47 *
[Matching] Ep 003/20 | TrainLoss 0.8815 || Val@1 67.77 Val@5 100.00 || Test@1 68.33 Test@5 100.00 || Best(val@1) 67.77 *
[Matching] Ep 004/20 | TrainLoss 0.8194 || Val@1 68.63 Val@5 100.00 || Test@1 71.63 Test@5 100.00 || Best(val@1) 68.63 *
[Matching] Ep 005/20 | TrainLoss 0.7648 || Val@1 68.20 Val@5 100.00 || Test@1 69.77 Test@5 100.00 || Best(val@1) 68.63 
[Matching] Ep 006/20 | TrainLoss 0.7310 || Val@1 70.73 Val@5 100.00 || Test@1 72.40 Test@5 100.00 || Best(val@1) 70.73 *
[Matching] Ep 007/20 | TrainLoss 0.6967 || Val@1 71.47 Val@5 100.00 || Test@1 72.70 Test@5 100.00 || Best(val@1) 71.47 *
[Matching] Ep 008/20 | TrainLoss 0.6728 || Val@1 74.47 Val@5 100.00 || Test@1 73.93 Test@

In [10]:
# ===== Matching Networks — EVAL-ONLY (independent cell) =====
import torch, numpy as np, time
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Subset, Sampler, DataLoader
from torchvision import datasets, transforms
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN = (0.4914, 0.4822, 0.4465); STD=(0.2470, 0.2435, 0.2616)
_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

# Data & splits
train_full = datasets.CIFAR10("./data", train=True,  download=True, transform=_tf)
test_set   = datasets.CIFAR10("./data", train=False, download=True, transform=_tf)
torch.manual_seed(0)
perm = torch.randperm(len(train_full))
val_idx = perm[:5000]
val_set = Subset(train_full, val_idx.tolist())

# Encoder
class Conv4(nn.Module):
    def __init__(self, out_dim=64):
        super().__init__()
        def block(i,o):
            return nn.Sequential(nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(True), nn.MaxPool2d(2))
        self.enc = nn.Sequential(block(3,32), block(32,64), block(64,64), block(64,out_dim))
        self.out_dim = out_dim
    def forward(self, x): return self.enc(x).mean(dim=[2,3])

# Episodic loader
class EpisodeSampler(Sampler):
    def __init__(self, base_set, n_way=5, k_shot=5, q_query=15, episodes_per_epoch=100):
        if isinstance(base_set, Subset):
            targets_full = np.array(base_set.dataset.targets)
            targets = targets_full[base_set.indices]
        else:
            targets = np.array(getattr(base_set, 'targets'))
        self.base = base_set; self.targets = targets
        self.n_way, self.k_shot, self.q_query = n_way, k_shot, q_query
        self.episodes_per_epoch = episodes_per_epoch
        self.class2idx = {c: np.where(self.targets==c)[0] for c in np.unique(self.targets)}
    def __len__(self): return self.episodes_per_epoch * self.n_way * (self.k_shot + self.q_query)
    def __iter__(self):
        for _ in range(self.episodes_per_epoch):
            classes = np.random.choice(list(self.class2idx.keys()), self.n_way, replace=False)
            ep_idx=[]
            for c in classes:
                pool = self.class2idx[c]
                if len(pool) < self.k_shot + self.q_query:
                    pool = np.random.choice(pool, self.k_shot + self.q_query, replace=True)
                else:
                    pool = np.random.permutation(pool)[:self.k_shot + self.q_query]
                ep_idx += pool.tolist()
            yield from ep_idx

def make_episode_loader(base_set, n_way, k_shot, q_query, episodes, num_workers=2):
    sampler = EpisodeSampler(base_set, n_way=n_way, k_shot=k_shot, q_query=q_query, episodes_per_epoch=episodes)
    bs = n_way*(k_shot+q_query)
    return DataLoader(base_set, batch_size=bs, sampler=sampler, num_workers=num_workers, pin_memory=True)

# Matching logits (cosine attention)
def logits_matching_cosine(q_feats, s_feats, s_labels, n_way, alpha=10.0):
    q = F.normalize(q_feats, dim=1); s = F.normalize(s_feats, dim=1)
    sim = q @ s.t()
    attn = F.softmax(alpha*sim, dim=1)
    onehot = F.one_hot(s_labels, num_classes=n_way).float()
    probs = attn @ onehot
    return torch.log(torch.clamp(probs, 1e-9, 1.0))

@torch.no_grad()
def eval_episodic_topk_matching(encoder, base_set, n_way, k_shot, q_query, episodes, alpha=10.0):
    encoder = encoder.to(device).eval()
    loader = make_episode_loader(base_set, n_way, k_shot, q_query, episodes)
    accs1=[]; accs5=[]
    topk = min(5, n_way)
    for xs,_ in loader:
        xs = xs.to(device)
        emb = encoder(xs)
        s_feats=[]; s_lbl=[]; q_feats=[]; q_lbl=[]
        for c in range(n_way):
            s = c*(k_shot+q_query)
            s_idx = slice(s, s+k_shot); q_idx = slice(s+k_shot, s+k_shot+q_query)
            s_feats.append(emb[s_idx]); s_lbl.append(torch.full((k_shot,), c, device=device, dtype=torch.long))
            q_feats.append(emb[q_idx]); q_lbl.append(torch.full((q_query,), c, device=device, dtype=torch.long))
        s_feats = torch.cat(s_feats,0); s_lbl = torch.cat(s_lbl,0)
        q_feats = torch.cat(q_feats,0); q_lbl = torch.cat(q_lbl,0)

        logp = logits_matching_cosine(q_feats, s_feats, s_lbl, n_way=n_way, alpha=alpha)
        _, pred = logp.topk(topk, 1, True, True)
        correct = pred.eq(q_lbl.view(-1,1))
        accs1.append(100.0*correct[:, :1].sum().item()/q_lbl.size(0))
        accs5.append(100.0*correct[:, :topk].sum().item()/q_lbl.size(0))
    return float(np.mean(accs1)), float(np.mean(accs5))

def matching_eval_from_ckpt(ckpt_path:str, n_way=5, k_shot=5, q_query=15, episodes=200, alpha=10.0):
    payload = torch.load(ckpt_path, map_location='cpu')
    meta = payload.get('meta', {})
    enc_dim = meta.get('enc_dim', 64)
    encoder = Conv4(out_dim=enc_dim).to(device)
    encoder.load_state_dict(payload['model_state'], strict=False)

    val_t1,val_t5 = eval_episodic_topk_matching(encoder, val_set,  n_way, k_shot, q_query, episodes=100, alpha=alpha)
    tst_t1,tst_t5 = eval_episodic_topk_matching(encoder, test_set, n_way, k_shot, q_query, episodes=episodes, alpha=alpha)
    print(f"[EVAL-Matching] {Path(ckpt_path).name} | Val Top1 {val_t1:.2f} Top5 {val_t5:.2f} || "
          f"Test Top1 {tst_t1:.2f} Top5 {tst_t5:.2f}")
    return (val_t1,val_t5),(tst_t1,tst_t5)

# ví dụ:
# matching_eval_from_ckpt("checkpoints/fsl_matching_networks_cosine.pth", n_way=5, k_shot=5, q_query=15, episodes=200, alpha=10.0)
